# Module 1:  Agentic RAG
## Part 2:  Agents

In Part 1, we built a working RAG pipeline with keyword search from scratch.  This is a fixed pipeline that has three steps:  
1. Search the FAQ
2. Build a prompt with the results
3. Send it to the LLM for it to give a helpful reply.

It gives reasonably good results as long as the user's query matches text in the FAQ documents.  However, if the user's request contains a typo in a key word, or phrases the question in an unsual way, or requires information that could only be obtained through multiple searches, then this system breaks down.

Rather than building a prompt with the results of our search of the FAQ database and sending it to the LLM in the last step, we can put the LLM in charge of the search process.  With the LLM "in charge", it can:
* fix typos
* search again with different terms
* ask the use clarifying questions

Giving the LLM the ability to manage the process makes this system __*agentic*__.  An agent uses an LLM to decide which actions to take, and in what order.  Part 2 of this module covers key aspects of this structure, including:  
1. Function calling (giving the LLM the ability to use tools to solve the problem),
2. The agentic loop (in which the LLM decides when to call a tool, when to call another if needed, and when to stop and answer), and
3. Frameworks (the libraries that run this loop for us)

This part of the module builds on the RAG pipeline we constructed in Part 1.  


### Quick RAG revision

As an example, we can see what happens when our query contains a typo.  We'll set up the RAG pipeline from Part 1 using our helper functions, ```ingest.py``` and ```rag_helper.py```.

First we need to load the OpenAI client.  We pull the necessary information, including the API key, from the environment file (this file is not committed to GitHub).  

In [39]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Load the data and build the search index:

In [40]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

Create the assistant:

In [41]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

#### Testing it

Let's try a question:

In [42]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally:\n\n1. Install Ollama from https://ollama.com/download for your operating system:\n   - macOS: download the `.pkg`\n   - Windows: download the `.msi`\n   - Linux: run\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. In a terminal, start a model locally with:\n   ```bash\n   ollama run llama3\n   ```\n   This downloads the LLaMA 3 model, starts it locally, and opens a chat-like interface.\n\n3. To test the local Ollama server, run:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a response like:\n   ```json\n   {"models": [...]}  \n   ```\n\nIf you want to use it from Python, install the client with:\n```bash\npip install ollama\n```\n\nThen you can call it with `ollama.chat(...)`.'

We can see that this works fine, giving us the kind of answer we would expect.  But what is there is a typo in the request?

In [43]:
assistant.rag("How do I run Olama locally?")

'I don’t see any FAQ entry about running **Ollama** locally.\n\nThe closest relevant guidance in the FAQ is that you can run the course locally if you’re comfortable setting up the needed tools like Python, `uv`, Jupyter, Docker, and any other module-specific tools. If you want, I can help you figure out a local setup based on that.'

Because the term 'Olama' is not contained in the database, the LLM can't produce a result.  This is because the lexical search looks for the exact word.  This is where an agent can be really helpful.

### The agentic approach

An agent puts the LLM in charge.

Instead of running search ourselves, we give the LLM a search tool. It decides when to call it and what to search for.

Now, instead of that exact question going into our pipeline and returning nothing, our pipeline process looks something like this:  
1. User question:  "How do I run Olama?"
2. LLM searches for 'Olama'
3. LLM turns up nothing in the database
4. LLM can infer that 'Olama' may actually refer to 'Ollama'
5. LLM runs the search again with "Ollama"
6. LLM returns helpful results!

The difference is about who makes the decisions.  

* With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
* With an agent, the LLM decides. It chooses which actions to take and when to stop.

The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

#### Asking without tools

First, let's see what the LLM does without any tools. We ask it a course-specific question and look at the answer.

In [44]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Yes—you can likely join it, but it depends on the course’s enrollment rules and whether registration is still open.\n\nIf you want, I can help you figure it out. Please send me:\n- the course name\n- the school/platform\n- when it starts\n- whether it’s in-person or online\n\nIf you’re asking the course organizer, you could say:\n> Hi, I just discovered the course and I’m very interested in joining. Is it still possible to enroll?'

Without any context, the LLM provides a genertic answer.  This is why we need RAG, and why we want to allow the model to use tools to provide useful results.

### Defining the tools

First we define a top-level ```search``` function that queries ```index``` directly. The model will reference it by this name. We keep the Python function and the tool name aligned so the dispatch is easier later.

In [45]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

Next we tell the model about this function. The model doesn't see our Python code, only a schema describing what the function does and what arguments it takes. LLMs are language agnostic. At the end we're just making an HTTP call, so we describe the tool in JSON rather than in Python. The same schema would work from TypeScript or Java.

In [46]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

The ```description``` is the most important field, because the model reads it to decide when to call the function. ```parameters``` is a JSON schema for the arguments, and we mark ```query``` as required so the model always fills it in.

#### Sending the question with the tool

Now we send the same question as before, but this time we include the tool in the request:

In [47]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"join course discovered late enrollment can I still join"}', call_id='call_y4RcKxRys2b6vWtwszpdfxb6', name='search', type='function_call', id='fc_068cdd426a095cfa006a39c32904bc819a9fa0f56131ad1761', namespace=None, status='completed')]

 Looking at the output, we see that the response contains a function_call entry. The model decided it needs to search the FAQ before answering. The models asks us to run the search function first.

Looking at the arguments too, we see something interesting.  The model didn't pass our question verbatim. It judged the raw question wasn't the best query to search with, so it rewrote our enrollment question into search keywords like "enroll late join course".

#### Executing the function and sending the result back

The function call contains JSON arguments. We parse them, call our ```search``` function, and serialize the result.

In [48]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

Now we send this result back to the model. First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result.

In [49]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

The ```call_id``` links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own ```call_id```.

#### Asking the model again

We call the API a second time with the expanded history:

In [50]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes — you can join even if you just discovered the course.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

Hooray!  We get a DataTalksClub course-specific answer.  We see the original question, the model's decision to call ```search```, and the FAQ results.

Note that we have to send the whole history, because LLMs are stateless between API calls--meaning it doesn't 'remember' the previous information we might already have sent to it. The memory is the list you send as ```input```. If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. That means the question, the decision to call search, and the result we got back.

That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

This pattern is referred to as "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same: the LLM decides which tools to call.

#### Token usage and cost

We just made two API calls instead of one. Each call we send to the model costs money, so it's worth checking how much one tool-using turn actually costs.

The response has a usage field with the token counts:



In [51]:
usage = response.usage
usage.input_tokens, usage.output_tokens

(810, 35)

Model providers publish prices for each model per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.

In [52]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


Note that this usage is only for the second API call. The first call also has its own usage and its own cost. That was the call where the model decided to invoke ```search```. Two calls means we pay twice. We pay even more on the second call, because we resend the full history as input.

With a real agent loop the model can make many calls, so the costs add up. It's a good idea to periodically check on ```usage``` as we develop apps.

### The Agentic Loop

While this pipeline is an improvement on the fixed pipeline from Part 1, it only allows one function call--one iteration.  What if the first search misses the answer, or if the model would run more than one function call if it had the ability to do so?  To allow this, we need a loop that keeps calling the model and running tools until it's done.  This is what an agent does.


#### Anatomy of an agent

An LLM that is able to make decisions to best complete the task at hand is an agent. It's an AI assistant whose goal is to help the user.

An agent has three parts:

1.  *Instructions*, the role and behavior we want. We pass this as the ```developer``` message. The better the instructions, the better the agent helps.
2.  *Tools*, the functions the agent can call to carry out the task. For us that's only ```search```.
3.  *Memory*, the message history. We append every prompt, every model output, and every tool result. The agent reads this to know what it has already tried.

Everything below is the code that wires these three together inside a loop.

#### A developer prompt

So far we've relied on the model to figure out when to search. We can make this more reliable with a ```developer``` message that spells out how to conduct the task.  The same message also pushes it toward multiple searches, so we get to watch the loop run more than once.

In [53]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

#### A function-call helper

We'll be running function calls repeatedly inside the loop, so let's wrap that in a small helper. It turns the JSON arguments into a Python dict, calls the right function, and serializes the result. We only have one tool for now, so we dispatch on the function name directly.

In [54]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

The helper returns the exact structure the Responses API expects. When we add more tools later, we'll extend this with more ```if``` branches (or switch to a registry).

#### Processing one response

Let's process a single model response. We append each output entry to the conversation, print any messages, and run any function calls. Function-call results get appended too.

In [55]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

messages.extend(response.output)
has_function_calls = False

for item in response.output:
    if item.type == "function_call":
        print("function_call:", item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)
        has_function_calls = True

    elif item.type == "message":
        print("ASSISTANT:")
        print(item.content[0].text)

function_call: search {"query":"join course late enrollment discovered course can I join"}
function_call: search {"query":"course enrollment can I still join if just discovered the course"}


The ```has_function_calls``` flag tells us whether the model needs another API call. If the response contains a function call, the updated ```messages``` has tool output the model hasn't seen yet. We'll need to send it back.

#### The full agent loop

We wrap this in a ```while``` loop.  The loop keeps calling the model until it returns a response without any function calls. We also keep an iteration counter so we can see how many round-trips happened.

In [56]:
it = 1

while True:
    print(f"iteration #{it}...")
    has_function_calls = False

    response = openai_client.responses.create(
        model="gpt-5.4-mini",
        input=messages,
        tools=[search_tool],
    )

    messages.extend(response.output)

    for item in response.output:
        if item.type == "function_call":
            print("function_call:", item.name, item.arguments)
            call_output = make_call(item)
            messages.append(call_output)
            has_function_calls = True

        elif item.type == "message":
            print("ASSISTANT:")
            print(item.content[0].text)

    it = it + 1
    if has_function_calls == False:
        break

iteration #1...
ASSISTANT:
Yes — you can still join the course after discovering it.

A couple of important notes:
- You can follow the materials and start learning anytime.
- If you want a certificate, you need to submit your project while submissions are still open.
- Certificates are only available for the live cohort, not self-paced study.

If you want, I can also help you figure out the best way to catch up quickly. Are there other areas you’d like to explore?


This is the core agent loop. The model reasons about the next action.  The code performs it, and the model sees the result on the next turn.  The loop stops when the model returns a final answer with no more tool calls.

The model decides how many times it searches, and we keep looping until it stops asking for tools.

The exit condition is the simplest one possible.  If the model doesn't call a function in a turn, we're done.  Other frameworks add safety nets on top, like a max iteration count, a token budget, or a wall-clock limit. We could cap it at, say, five iterations and force an answer on the last one. The core is still this one flag.

#### Wrapping it in a function

Let's wrap the loop in a function so we can reuse it. The function takes the instructions and the question as parameters, and returns the final answer.

In [57]:
def agent_loop(instructions, question, model="gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break

    return last_answer

Let's try it with a question that has a typo:

In [58]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Ollama local run install start model pull run localhost FAQ"}
function_call: search {"query":"run Ollama locally command ollama serve ollama run FAQ"}
iteration #2...
ASSISTANT:
To run **Ollama locally**:

1. **Install Ollama**
   - macOS: download the installer from https://ollama.com/download
   - Windows: download the `.msi`
   - Linux:
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. **Start a model locally**
   ```bash
   ollama run llama3
   ```
   This will download the model and open a local chat interface.

3. **Check that the local server is running**
   ```bash
   curl http://localhost:11434
   ```
   If it’s working, you should get a response from the Ollama server.

4. **If you need to restart the server**
   ```bash
   nohup ollama serve > nohup.out 2>&1 &
   ```

5. **Optional: use it from Python**
   ```bash
   pip install ollama
   ```

   ```python
   import ollama

   response = ollama.chat(


'To run **Ollama locally**:\n\n1. **Install Ollama**\n   - macOS: download the installer from https://ollama.com/download\n   - Windows: download the `.msi`\n   - Linux:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Start a model locally**\n   ```bash\n   ollama run llama3\n   ```\n   This will download the model and open a local chat interface.\n\n3. **Check that the local server is running**\n   ```bash\n   curl http://localhost:11434\n   ```\n   If it’s working, you should get a response from the Ollama server.\n\n4. **If you need to restart the server**\n   ```bash\n   nohup ollama serve > nohup.out 2>&1 &\n   ```\n\n5. **Optional: use it from Python**\n   ```bash\n   pip install ollama\n   ```\n\n   ```python\n   import ollama\n\n   response = ollama.chat(\n       model=\'llama3\',\n       messages=[{"role": "user", "content": "Hello!"}]\n   )\n\n   print(response[\'message\'][\'content\'])\n   ```\n\nIf you want, I can also show you how to us

We can see what happens at every step. The agent searches for "Olama" and doesn't get good results.  It then searches again with "Ollama" and finds the answer.  The loop allows the model recover from a bad search on its own. 

Also try the course enrollment question:

In [59]:
agent_loop(instructions, "I just discovered the course. Can I still join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I still join"}
function_call: search {"query":"course enrollment late join start date registration FAQ"}
function_call: search {"query":"I just discovered the course can I still join FAQ"}
iteration #2...
ASSISTANT:
Yes — you can still join the course. You can start learning and follow the materials even if you discovered it late.

A couple of important notes:
- If you want a certificate, you need to submit your project while submissions are still open.
- If the course is currently running, you can also submit homework while the forms are open.

If you want, I can also help you figure out the best way to catch up quickly.


'Yes — you can still join the course. You can start learning and follow the materials even if you discovered it late.\n\nA couple of important notes:\n- If you want a certificate, you need to submit your project while submissions are still open.\n- If the course is currently running, you can also submit homework while the forms are open.\n\nIf you want, I can also help you figure out the best way to catch up quickly.'

We get good results here as well:  DataTalksClub-specific course information that answers the question.

### Encouraging multiple searches

There's a subtle issue here. The model often answers after the first search, even when more searches would help. It reasons that it already knows enough, so it doesn't run additional searches.  Let's push it to explore more by rewriting the instructions.

In [60]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I join FAQ"}
iteration #2...
function_call: search {"query":"certificate submit project while accepting submissions peer review live cohort self-paced course FAQ"}
iteration #3...
ASSISTANT:
Yes — you can still join the course.

A couple of important notes:
- You can start learning and following the materials even if you discovered it late.
- If you want a certificate, you need to submit your project while submissions are still open, and the course certificate is only available for the live cohort, not self-paced.

If you want, I can also help you figure out how to start catching up quickly. Are there other areas you want to explore?


'Yes — you can still join the course.\n\nA couple of important notes:\n- You can start learning and following the materials even if you discovered it late.\n- If you want a certificate, you need to submit your project while submissions are still open, and the course certificate is only available for the live cohort, not self-paced.\n\nIf you want, I can also help you figure out how to start catching up quickly. Are there other areas you want to explore?'

Now the agent makes multiple searches per question and doesn't stop after the first round of results. The **_instructions_** are how we steer the agent. It can still decide to skip ahead, though, so don't expect it to follow them every single run.

#### Restricting off-topic questions

Right now the agent will try to answer anything we ask it.  For example, we could ask it a question about chess and it will attempt to answer:

In [61]:
agent_loop(instructions, "What is the Queen's Gambit?")

iteration #1...
function_call: search {"query":"Queen's Gambit definition chess opening course FAQ"}
iteration #2...
function_call: search {"query":"Queen's Gambit chess opening what is it explanation"}
iteration #3...
ASSISTANT:
The Queen’s Gambit is a **chess opening** that starts with the moves:

1. d4 d5  
2. c4

In this opening, White offers a pawn on c4 to try to **gain control of the center** and create a more active position. If Black accepts the pawn, it’s called the **Queen’s Gambit Accepted**; if Black doesn’t, it’s the **Queen’s Gambit Declined**.

If you want, I can also explain:
- why it’s called a “gambit,”
- the main ideas for White and Black,
- or show a few common lines.


'The Queen’s Gambit is a **chess opening** that starts with the moves:\n\n1. d4 d5  \n2. c4\n\nIn this opening, White offers a pawn on c4 to try to **gain control of the center** and create a more active position. If Black accepts the pawn, it’s called the **Queen’s Gambit Accepted**; if Black doesn’t, it’s the **Queen’s Gambit Declined**.\n\nIf you want, I can also explain:\n- why it’s called a “gambit,”\n- the main ideas for White and Black,\n- or show a few common lines.'

We don't want to expend resources answering questions that don't pertain to our courses, so we can modify our instructions to the LLM to only answer questions from the course FAQ.  

In [62]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit chess opening queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit queen gambit course faq"}
iteration #3...
ASSISTANT:
I couldn’t find any course FAQ entry about “queen gambit,” so this seems off-topic for the course.

If you meant something else related to the course, feel free to clarify. Otherwise, is there another course-related area you want to explore?


'I couldn’t find any course FAQ entry about “queen gambit,” so this seems off-topic for the course.\n\nIf you meant something else related to the course, feel free to clarify. Otherwise, is there another course-related area you want to explore?'

This is a lightweight form of an input guardrail:  we tell the agent what's in scope and what isn't.  A *real* guardrail checks the input before the agent runs and can block off-topic questions outright. While we aren't looking to implement real guardrails right now, we see that instructions are the first place to start with any kind of guardrail.

This handwritten loop is the best way to understand what frameworks don't automatically reveal. Every agent framework wraps this same pattern, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.

### ToyAIKit

Writing the agent loop by hand is educational but tedious.  We don't want to have to do this every time.  We'd like to have a toolkit that can quickly do this for us, so we can focus on prompts, tools, and behaviors.  

Alexey built ToyAIKit in a DTC workshop a while back.  It's small and easy to read, so it's useful for developing and debugging locally and learning the basics in this course.  Note:  ToyAIKit is an experimental library built for educational purposes, and is not intended for use in production.  

#### Setup

Install it:

In [63]:
# !uv add toyaikit


Import the classes we need:

In [64]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

#### Registering the tool

We register our ```search``` function along with the schema from earlier lessons:



In [65]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

#### Letting ToyAIKit generate the schema

Writing that schema by hand is annoying, and we don't want to do it for every function.  Luckily, we don't have to.

If we add a type hint and a docstring to ```search```, ToyAIKit reads them and derives the schema for us:

In [66]:
def search(query: str) -> dict[str, str]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )

Then register it without passing a schema:

In [67]:
agent_tools = Tools()
agent_tools.add_tool(search)

Let's see what ToyAIKit produced:

In [68]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the FAQ database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

The output is the same JSON schema we hand-wrote in the function calling lesson. ToyAIKit generated it from the docstring and the type hint.

Every modern agent framework does this same trick. It reads a typed Python function with a docstring and builds the schema from it. The OpenAI Agents SDK, PydanticAI, LangChain and Google ADK all work this way. You write the tool and the framework figures out how to describe it.



#### The chat interface and runner

Create the chat interface and a callback, then build the runner:

In [69]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The ```chat_interface``` handles display in the notebook. The ```callback``` renders model messages and tool calls as they happen. The ```runner``` runs the agent loop, the same ```while True``` we wrote by hand. It sends messages, executes function calls, adds tool outputs back, and repeats until the model is done.

We pick ```gpt-5.4-mini``` here on purpose. Without it, ToyAIKit falls back to a smaller, faster default that doesn't follow the instructions as reliably.



#### Running one prompt
Run a single prompt:

In [70]:
result = runner.loop(
    prompt="How do I run Olama locally?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received


We used the typo "Olama" on purpose. The agent searches and gets poor results, then retries with "Ollama". The recovery is the same as the handwritten loop. The notebook output is nicer to watch. Each tool call and message renders inline, so you can look at every search result.

The ```result``` is a ```LoopResult``` with ```all_messages``` (the full conversation), token counts, and ```cost``` (computed from token usage).



#### Cost and tokens
Let's find out what the call cost:

In [71]:
result.cost

CostInfo(input_cost=Decimal('0.0027555'), output_cost=Decimal('0.0013365'), total_cost=Decimal('0.0040920'))

This is useful while developing--especially with multi-turn agents where one prompt can trigger several model calls. The handwritten loop made you compute this by hand. The framework keeps a running total for you.

You can also look at the full message history:

In [72]:
result.all_messages

[EasyInputMessage(content="You're a course teaching assistant.\nYou're given a question from a course student and your task is to answer it.\n\nIf you want to look up information, use the search function. \nUse as many keywords from the user question as possible when making first requests.\n\nMake multiple searches. First perform search, analyze the results \nand then perform more searches. \n\nThe question has to be about the course or its logistics, offtopic questions \nshouldn't be answered. If the search returns nothing, it's likely an off-topic question.\nIf you can't answer the question using FAQ, don't do it yourself. Only use the \nfacts from the FAQ database.\n\nAt the end, ask if there are other areas that the user wants to explore.", role='developer', phase=None, type=None),
 EasyInputMessage(content='How do I run Olama locally?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"Olama local run Ollama locally"}', call_id='call_j6KPpUT1htYfn

This is just a list - the same ```messages``` list we maintained by hand.

#### Continuing the conversation
Take the messages from the previous result and pass them as ```previous_messages``` on the next ```loop``` call:

In [73]:
result2 = runner.loop(
    prompt="How do I run a different model?",
    previous_messages=result.all_messages,
    callback=callback,
)

-> Response received


-> Response received


The runner picks up where the last call left off, with the same agent loop and an extended history. The model knows "different model" refers to Ollama because it sees the previous turn in memory. Without that history, it would have no idea what we're asking about.

#### Interactive chat
For a chat-like workflow, run the built-in input loop, typing questions in the chat box that appears on screen.  (To exit the interactive chat, type "stop".)

In [74]:
runner.run();

# In VS Code, the interactive chat window is at the top of the Jupyter Notebook window.


-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


### Other Frameworks

As mentioned at the end of the section right before the ToyAIKit section, the agentic loop pipeline we set up follows essentially the same pattern--function calling, the agent loop, and tool definitions--as every other agent framework, whether it's LangChain, PydanticAI, or the OpenAI Agents SDK.  We now understand the basics of how any production framework works.  This module was framework agnostic, but it's worth knowing a little bit more about the frameworks mentioned above and how to install them.  

#### OpenAI Agents SDK
This is the official SDK from OpenAI for building agents. It uses the same Responses API we used throughout this module. It supports tool definition, multi-turn conversations, and handoffs between agents.  It's a good choice if you're already using OpenAI and want something official and well-maintained. To install using the ```uv``` package manager:

```uv add openai-agents```


#### PydanticAI
Alexey describes PydanticAI as a type-safe agent framework that supports multiple LLM providers.  (Type-safe agents are designed to strictly adhere to predefined schema or a set of data types.)  Tools are plain Python functions with type hints.  No wrappers are needed. Switching providers is as simple as changing the model string.

```uv add pydantic-ai```

This is Alexey's favorite.  While he appreciates the type-safety of PydanticAI, he says that other frameworks offer this as well.  The main reasons he likes it are the usability of PydanticAI and the team behind it.  He says it's a good choice if you want type safety and multi-provider support.

(I decided to look up some additional information about type-safe agents generally and PydanticAI more specifically.  A Google search summary turned up Mastra and DSPy, in addition to PydanticAI.  Looking at PydanticAI's website specifically, I learned that Pydantic Validation is the validation layer of the OpenAI SDK, the Google ADK, the Anthropic SDK, LangChain, LlamaIndex, AutoGPT, Transformers, CrewAI, Instructor and many more.  As Pydantic says, "why use the derivative when you can go straight to the source?" Regarding type-safety, Pydantic states that PydanticAI is "...[designed] to give your IDE or AI coding agent as much context as possible for auto-completion and type checking, moving entire classes of errors from runtime to write-time for a bit of that Rust "if it compiles, it works" feel.")


#### LangChain / LangGraph
A popular framework with lots of integrations. LangChain handles the basics, and LangGraph adds graph-based workflows for more complex agent patterns.

Good choice if you need lots of integrations (vector stores, document loaders, etc.) and a large community.

#### Google ADK
The Agent Development Kit from Google. It exposes the same building blocks we've seen, like tools, instructions, and sessions. It also integrates with Google Cloud.  Best choice if you plan to use Gemini models and/or if your stack is on Google Cloud.

#### Others
Here are some other frameworks worth knowing about:

CrewAI - multi-agent orchestration
AutoGen - multi-agent conversations from Microsoft
Semantic Kernel - from Microsoft, supports C# and Python
Smolagents - lightweight agent framework from HuggingFace
Anthropic Tool Use - Anthropic's native tool use API

Pick one that fits your stack and your needs. The hard part is designing good tools and prompts - the loop is always the same.

### A note about avoiding agents when a simpler tool will do the job

We just spent some time demonstrating a use case where agentic AI can be very useful.  That said, agents aren't always the best tool for the job, for a number of reasons:

* Cost:  There are likely to be more API calls per request because the loop can initiate many tool calls before the model is satisfied; furthermore, each iteration is another billed call that sends the full message history each time
* Time:  Because there may be many "round trips", each of which the model must complete before moving on to the next one, agentic approaches may involve a lot of lag time
* Monitoring/cognitive load:  Need to monitor cost, iteration count, and whether the agent is actually solving the problem or going in circles 
* Less predictable behavior:  LLMs are non-deterministic; the LLM can make different decisions on the same prompt run two different times, with different resulting paths

As with any problem-solving activity, the first step is figuring which tool is best for the job.  Many tasks can be accomplished by simpler approaches that are less costly (in terms of time, money, and monitoring / tracking results).  For example, before using an agentic approach, we should consider the following alternatives:

* Plain RAG--one search, one answer
* Parsing or templating a document into another form
* A single LLM call with no tools

If a simpler approach works well for your problem, use that.  Only if simpler approaches don't solve your problem should you reach for an agent loop.  Then you'll be more sure that you've selected the right tool for the job and are not needlessly wasting resources implementing it.